# FarmTech Vision | Visão computacional com YOLO e CNN

**FIAP | Inteligência Artificial | Fase 6, Capítulo 1: O despertar da Rede Neural**

| | |
|---|---|
| **Aluno** | Felipe Yamabe Shimoda de Lima |
| **RM** | 573344 |
| **Turma** | 1TIAOB |

## Sumário

1. Contexto e objetivo
2. Preparação do ambiente
3. Parte 1 | Entrega 1: YOLOv5 customizada
    - 3.1 Dataset
    - 3.2 Rotulação no Make Sense
    - 3.3 Configuração do dataset
    - 3.4 Treinamento com 30 épocas
    - 3.5 Treinamento com 60 épocas
    - 3.6 Comparação entre 30 e 60 épocas
    - 3.7 Validação
    - 3.8 Teste
    - 3.9 Conclusões da Entrega 1
4. Parte 2 | Entrega 2: YOLO tradicional e CNN treinada do zero
    - 4.1 YOLO tradicional
    - 4.2 CNN treinada do zero
    - 4.3 Tempo de inferência
    - 4.4 Comparação consolidada
    - 4.5 Facilidade de uso e integração
    - 4.6 Análise crítica
5. Conclusões finais
6. Referências

## 1. Contexto e objetivo

A FarmTech Solutions ampliou sua carteira para além do agronegócio: saúde animal, segurança patrimonial de fazendas e residências, controle de acesso de funcionários, análise de documentos e, agora, visão computacional. Em visita a um cliente que quer entender como um sistema desse tipo funciona na prática, o time de desenvolvimento precisa **demonstrar o potencial e a acurácia de um detector de objetos treinado sob medida**.

### Cenário escolhido: maçã e tesoura

| Critério | Por que maçã e tesoura |
|---|---|
| Objetos bem diferentes | A maçã é orgânica, arredondada e colorida; a tesoura é metálica ou plástica, alongada e tem partes vazadas |
| Aderência ao cliente | Remetem à rotina de uma fazenda: a produção colhida e as ferramentas de trabalho |
| Comparação justa na Entrega 2 | As duas classes estão entre as 80 classes da base COCO, usada no treino da YOLO tradicional. Assim, a YOLO pré-treinada consegue, em princípio, reconhecê-las sem nenhum treino adicional |

### Dataset

- 80 imagens, 40 por classe, reunidas a partir de bases públicas de imagens com licença livre (Open Images V7 e Wikimedia Commons), selecionadas para variar fundo, iluminação, ângulo, distância e tipo de objeto.
- Imagens da base COCO foram evitadas de propósito, para que a YOLO tradicional não fosse testada em fotos que já viu durante o seu treinamento.
- Divisão por classe: **32 imagens para treino, 4 para validação e 4 para teste**, sorteadas com semente fixa.
- Rotulação feita no **Make Sense**, com caixas delimitadoras exportadas no formato YOLO.

## 2. Preparação do ambiente

O notebook foi pensado para rodar no **Google Colab com GPU T4**, conectado ao Google Drive, e também funciona em uma máquina local.

| Etapa | No Colab | Localmente |
|---|---|---|
| Dataset | O repositório do projeto é clonado e a pasta `dataset` é organizada no Drive, em `MyDrive/FIAP/Fase6_Cap1_FarmTechVision` | Usa a pasta `dataset` ao lado do notebook |
| YOLOv5 | Clonado do repositório oficial da Ultralytics em um commit fixo | Idem |
| Resultados | Copiados para o Drive ao final | Ficam em `yolov5/runs` e `resultados` |

Estrutura do dataset no Drive, a mesma esperada pelo YOLOv5:

```
Fase6_Cap1_FarmTechVision/
├── farmtech.yaml
└── dataset/
    ├── images/
    │   ├── train/   64 imagens
    │   ├── val/      8 imagens
    │   └── test/     8 imagens
    └── labels/
        ├── train/   64 rótulos exportados do Make Sense
        ├── val/      8 rótulos
        └── test/     8 rótulos
```

In [ ]:
# Bibliotecas padrão do Python
import os
import sys
import time
import shutil
import random
import subprocess
from pathlib import Path

# Bibliotecas de dados, imagem e visualização
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont

# Semente fixa para tornar os experimentos reproduzíveis
SEMENTE = 42
random.seed(SEMENTE)
np.random.seed(SEMENTE)

# Classes na mesma ordem criada no Make Sense: 0 = maçã, 1 = tesoura
CLASSES = ["maca", "tesoura"]
NOMES_EXIBICAO = {"maca": "Maçã", "tesoura": "Tesoura"}
# Cor das caixas de cada classe nas imagens
CORES = {"maca": (232, 10, 110), "tesoura": (0, 150, 255)}

# Quantidade de épocas das duas simulações de treinamento pedidas no enunciado
EPOCAS = [30, 60]

# Limiares da avaliação: confiança mínima de uma detecção e sobreposição mínima com a caixa real
LIMIAR_CONFIANCA = 0.25
LIMIAR_IOU = 0.5

# Aparência padrão dos gráficos
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

In [ ]:
# Detecta se o notebook está rodando no Google Colab ou em uma máquina local
EM_COLAB = "google.colab" in sys.modules

# Repositório do projeto (dataset) e repositório oficial do YOLOv5 em versão fixa
REPO_URL = "https://github.com/dev-felipeshimoda/Fase-6---Cap-1---O-despertar-da-Rede-Neural.git"
YOLOV5_URL = "https://github.com/ultralytics/yolov5.git"
YOLOV5_COMMIT = "35b48237aef6d71ca9de2c5dea345d7536eb7fa7"

if EM_COLAB:
    # Conecta o Google Drive ao Colab (a conta Google pede autorização)
    from google.colab import drive
    drive.mount("/content/drive")
    RAIZ = Path("/content")
    PASTA_DRIVE = Path("/content/drive/MyDrive/FIAP/Fase6_Cap1_FarmTechVision")
    PASTA_DRIVE.mkdir(parents=True, exist_ok=True)
    # Clona o repositório do projeto, que contém as imagens e os rótulos exportados do Make Sense
    if not (RAIZ / "repo").exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, str(RAIZ / "repo")], check=True)
    DATASET = PASTA_DRIVE / "dataset"
    # Na primeira execução, organiza o dataset no Drive: images e labels, cada um com train, val e test
    if not DATASET.exists():
        shutil.copytree(RAIZ / "repo" / "dataset", DATASET, ignore=shutil.ignore_patterns("*.cache"))
else:
    # Execução local: o notebook fica na raiz do repositório, ao lado da pasta dataset
    RAIZ = Path.cwd()
    PASTA_DRIVE = None
    DATASET = RAIZ / "dataset"

# Pasta que guarda tabelas e figuras geradas pelo notebook
SAIDAS = (PASTA_DRIVE if EM_COLAB else RAIZ) / "resultados"
SAIDAS.mkdir(parents=True, exist_ok=True)

print(f"Ambiente: {'Google Colab' if EM_COLAB else 'local'}")
print(f"Dataset:  {DATASET}")
print(f"Saídas:   {SAIDAS}")

In [ ]:
# Pasta onde o repositório do YOLOv5 fica clonado
YOLOV5 = RAIZ / "yolov5"
if not YOLOV5.exists():
    # Clona o YOLOv5 e fixa o commit, evitando mudanças de versão entre execuções
    subprocess.run(["git", "clone", "-q", YOLOV5_URL, str(YOLOV5)], check=True)
    subprocess.run(["git", "-C", str(YOLOV5), "checkout", "-q", YOLOV5_COMMIT], check=True)

# Instala as dependências do YOLOv5, mantendo o OpenCV na série 4.x: a série 5 removeu o leitor
# de modelos Darknet, usado pela YOLOv3 tradicional na Parte 2
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(YOLOV5 / "requirements.txt"),
                "opencv-python<5"], check=True)

import torch

# Usa a GPU quando disponível (no Colab, a T4); caso contrário, a CPU
DISPOSITIVO = "0" if torch.cuda.is_available() else "cpu"
# Processos de leitura de dados: no Windows o valor 0 evita erros de multiprocessamento
TRABALHADORES = 2 if EM_COLAB else 0

commit = subprocess.run(["git", "-C", str(YOLOV5), "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print(f"PyTorch {torch.__version__}")
print(f"Dispositivo: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"YOLOv5 no commit {commit}")

In [ ]:
def executar(script, *argumentos):
    '''Executa um script do YOLOv5 em subprocesso, mostra a saída e devolve a duração em segundos.'''
    # Saída em UTF-8 e integrações externas de registro desligadas
    ambiente = {**os.environ, "PYTHONIOENCODING": "utf-8", "WANDB_MODE": "disabled", "COMET_MODE": "disabled"}
    comando = [sys.executable, str(YOLOV5 / script), *map(str, argumentos)]
    inicio = time.perf_counter()
    processo = subprocess.Popen(comando, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                                encoding="utf-8", errors="replace", env=ambiente, cwd=RAIZ)
    for linha in processo.stdout:
        # Barras de progresso reescrevem a mesma linha com \r: mantém só a versão final
        final = linha.rstrip("\n").split("\r")[-1]
        if final.strip():
            print(final)
    processo.wait()
    if processo.returncode != 0:
        raise RuntimeError(f"{script} terminou com código {processo.returncode}")
    return time.perf_counter() - inicio

# 3. Parte 1 | Entrega 1: YOLOv5 customizada

A proposta é ensinar uma rede YOLOv5 a reconhecer maçãs e tesouras a partir de poucas imagens. Em vez de começar do zero, o treino parte da **YOLOv5s pré-treinada na base COCO** e ajusta seus pesos às nossas duas classes (*transfer learning*), como no capítulo sobre ESP32 e visão computacional.

O trabalho segue três etapas:

| Etapa | Imagens | O que acontece | Script do YOLOv5 |
|---|---|---|---|
| **Treinamento** | 64 (treino) | A rede ajusta os pesos época a época, comparando suas previsões com as caixas rotuladas | `train.py` |
| **Validação** | 8 (validação) | Ao fim de cada época, a rede é medida em imagens que não usa para aprender; o melhor ponto vira o `best.pt` | `train.py` e `val.py` |
| **Teste** | 8 (teste) | O modelo final é aplicado em imagens nunca vistas, gerando as imagens com as detecções | `detect.py` |

### 3.1 Dataset

Contagem de imagens por classe e por conjunto, seguida de uma amostra do treino com as caixas desenhadas a partir dos rótulos.

In [ ]:
def ler_rotulos(caminho):
    '''Lê um rótulo YOLO e devolve a lista de objetos (classe, x_centro, y_centro, largura, altura).'''
    if not caminho.exists():
        return []
    partes = (linha.split() for linha in caminho.read_text().splitlines())
    return [(int(p[0]), *map(float, p[1:])) for p in partes if len(p) == 5]

SPLITS = ["train", "val", "test"]
NOMES_SPLIT = {"train": "Treino", "val": "Validação", "test": "Teste"}

registros = []
for split in SPLITS:
    for imagem in sorted((DATASET / "images" / split).glob("*.jpg")):
        rotulos = ler_rotulos(DATASET / "labels" / split / f"{imagem.stem}.txt")
        largura, altura = Image.open(imagem).size
        registros.append({
            "split": split,
            "arquivo": imagem.name,
            "classe": imagem.stem.split("_")[0],
            "objetos": len(rotulos),
            "largura": largura,
            "altura": altura,
            # Fração média da imagem ocupada pelas caixas daquela foto
            "area_caixa": np.mean([w * h for *_, w, h in rotulos]) if rotulos else 0.0,
        })
df_dataset = pd.DataFrame(registros)

# Tabela de imagens por classe e conjunto
tabela = pd.crosstab(df_dataset["classe"].map(NOMES_EXIBICAO), df_dataset["split"].map(NOMES_SPLIT),
                     margins=True, margins_name="Total")
tabela = tabela[["Treino", "Validação", "Teste", "Total"]]
tabela.index.name, tabela.columns.name = "Classe", None
display(tabela)

In [ ]:
# Resumo dos objetos rotulados por classe
resumo_dataset = df_dataset.groupby("classe").agg(
    imagens=("arquivo", "count"),
    objetos_rotulados=("objetos", "sum"),
    imagens_com_mais_de_um_objeto=("objetos", lambda s: int((s > 1).sum())),
    area_media_da_caixa=("area_caixa", "mean"),
)
resumo_dataset.index = resumo_dataset.index.map(NOMES_EXIBICAO)
resumo_dataset["area_media_da_caixa"] = (resumo_dataset["area_media_da_caixa"] * 100).round(1).astype(str) + "%"
display(resumo_dataset)

def desenhar_rotulos(caminho, rotulos, espessura=5):
    '''Desenha sobre a imagem as caixas de rótulos no formato YOLO.'''
    imagem = Image.open(caminho).convert("RGB")
    W, H = imagem.size
    desenho = ImageDraw.Draw(imagem)
    for classe, xc, yc, w, h in rotulos:
        caixa = [(xc - w / 2) * W, (yc - h / 2) * H, (xc + w / 2) * W, (yc + h / 2) * H]
        desenho.rectangle(caixa, outline=CORES[CLASSES[classe]], width=espessura)
    return imagem

def grade_imagens(imagens, titulos, colunas=4, tamanho=3.2, titulo_geral=None, arquivo=None):
    '''Exibe imagens PIL em grade e, se pedido, salva a figura.'''
    linhas = int(np.ceil(len(imagens) / colunas))
    figura, eixos = plt.subplots(linhas, colunas, figsize=(colunas * tamanho, linhas * tamanho), squeeze=False)
    for eixo in eixos.ravel():
        eixo.axis("off")
    for eixo, imagem, titulo in zip(eixos.ravel(), imagens, titulos):
        eixo.imshow(imagem)
        eixo.set_title(titulo, fontsize=9)
    if titulo_geral:
        figura.suptitle(titulo_geral, fontsize=13, fontweight="bold")
    plt.tight_layout()
    if arquivo:
        figura.savefig(SAIDAS / arquivo, bbox_inches="tight")
    plt.show()

# Amostra de 6 imagens de treino de cada classe, com as caixas rotuladas
amostra = pd.concat([df_dataset[(df_dataset.split == "train") & (df_dataset.classe == c)].sample(6, random_state=SEMENTE)
                     for c in CLASSES])
pasta_treino = DATASET / "images" / "train"
grade_imagens(
    [desenhar_rotulos(pasta_treino / a, ler_rotulos(DATASET / "labels" / "train" / a.replace(".jpg", ".txt")))
     for a in amostra["arquivo"]],
    list(amostra["arquivo"]), colunas=6, tamanho=2.6,
    titulo_geral="Amostra do treino com as caixas rotuladas no Make Sense", arquivo="amostra_dataset.png",
)

**[A PREENCHER COM OS RESULTADOS DA EXECUÇÃO]**

### 3.2 Rotulação no Make Sense

As 80 imagens foram rotuladas no [Make Sense](https://www.makesense.ai), ferramenta gratuita que roda no navegador:

1. **Get Started** e carregamento das imagens.
2. Projeto do tipo **Object Detection**, com as classes criadas nesta ordem: `maca` (0) e `tesoura` (1).
3. Um **retângulo** justo em volta de cada objeto, associado à sua classe. Polígonos não são usados, porque o YOLO trabalha com caixas.
4. **Actions > Export Annotations > YOLO format**, que gera um arquivo `.txt` por imagem, com o mesmo nome da foto.
5. Os rótulos foram guardados junto das imagens, em `labels/train`, `labels/val` e `labels/test`, e organizados no Google Drive.

Cada linha do arquivo descreve um objeto: `classe x_centro y_centro largura altura`, com valores normalizados entre 0 e 1 em relação ao tamanho da imagem. Exemplo de uma imagem de treino com duas maçãs:

In [ ]:
# Conteúdo de um rótulo exportado pelo Make Sense: uma linha por objeto
exemplo = DATASET / "labels" / "train" / "maca_02.txt"
print(exemplo.name)
print(exemplo.read_text())

### 3.3 Configuração do dataset

O YOLOv5 lê um arquivo YAML com o caminho das imagens de cada conjunto e o nome das classes. Os rótulos são encontrados automaticamente: para cada pasta `images/...`, o YOLOv5 procura a pasta equivalente `labels/...`.

In [ ]:
# Arquivo de configuração lido pelo YOLOv5: pasta do dataset, conjuntos e nomes das classes
ARQUIVO_YAML = (PASTA_DRIVE if EM_COLAB else RAIZ) / "farmtech.yaml"
ARQUIVO_YAML.write_text(
    f'path: "{DATASET.as_posix()}"\n'
    "train: images/train\n"
    "val: images/val\n"
    "test: images/test\n\n"
    "names:\n" + "".join(f"  {indice}: {nome}\n" for indice, nome in enumerate(CLASSES)),
    encoding="utf-8",
)
print(ARQUIVO_YAML.read_text(encoding="utf-8"))

### 3.4 Etapa de treinamento | Simulação 1: 30 épocas

Parâmetros usados nas duas simulações, que diferem **apenas no número de épocas**:

| Parâmetro | Valor | Motivo |
|---|---|---|
| `--weights` | `yolov5s.pt` | Versão *small* pré-treinada no COCO: rápida e adequada a um dataset pequeno |
| `--img` | 640 | Resolução padrão do YOLOv5, igual à das imagens preparadas |
| `--batch` | 16 | Cabe com folga na memória da GPU T4 |
| `--epochs` | 30 e 60 | Valores bem diferentes entre si, como pede o enunciado |
| `--seed` | 42 | Reprodutibilidade |
| `--cache ram` | ativo | As imagens ficam na memória e cada época fica mais rápida |

A cada época o YOLOv5 exibe as perdas de treino (`box_loss`: posição das caixas; `obj_loss`: presença de objeto; `cls_loss`: classe) e, ao final dela, valida o modelo nas 8 imagens de validação, mostrando precisão (P), recall (R), mAP50 e mAP50-95. Os pesos da melhor época são salvos em `weights/best.pt`.

In [ ]:
PASTA_TREINOS = YOLOV5 / "runs" / "train"
TREINOS = {}

def treinar(epocas):
    '''Treina a YOLOv5s a partir dos pesos do COCO e registra a pasta de resultados e o tempo gasto.'''
    nome = f"exp_{epocas}ep"
    duracao = executar(
        "train.py",
        "--img", 640,                # resolução de entrada
        "--batch", 16,               # imagens por passo de otimização
        "--epochs", epocas,          # passagens completas pelo conjunto de treino
        "--data", ARQUIVO_YAML,      # configuração do dataset
        "--weights", "yolov5s.pt",   # ponto de partida: YOLOv5 small pré-treinada no COCO
        "--device", DISPOSITIVO,     # GPU ou CPU
        "--workers", TRABALHADORES,  # processos de leitura de dados
        "--seed", SEMENTE,           # semente do treinamento
        "--cache", "ram",            # imagens em memória
        "--project", PASTA_TREINOS,  # pasta raiz dos treinos
        "--name", nome,              # subpasta desta simulação
        "--exist-ok",                # reaproveita a pasta se o notebook for executado de novo
    )
    TREINOS[epocas] = {"pasta": PASTA_TREINOS / nome, "segundos": duracao}
    print(f"\nTreino com {epocas} épocas concluído em {duracao / 60:.1f} min | resultados em {PASTA_TREINOS / nome}")

treinar(EPOCAS[0])

### 3.5 Etapa de treinamento | Simulação 2: 60 épocas

Mesmos parâmetros, com o dobro de épocas. A pergunta é se mais tempo de treino melhora o modelo ou se ele passa a decorar as imagens de treino (*overfitting*).

In [ ]:
# Segunda simulação: mesma configuração, com mais épocas
treinar(EPOCAS[1])

### 3.6 Comparação entre 30 e 60 épocas

Como ler as métricas:

| Métrica | O que mede | Referência |
|---|---|---|
| **Precisão (P)** | Das caixas que o modelo desenhou, quantas estavam certas | Quanto mais perto de 1, menos alarmes falsos |
| **Recall (R)** | Dos objetos reais, quantos o modelo encontrou | Quanto mais perto de 1, menos objetos perdidos |
| **mAP50** | Precisão média considerando certa a caixa com IoU de pelo menos 0,5 com a caixa real | Acima de 0,5 é bom; acima de 0,75, muito bom |
| **mAP50-95** | Média do mAP com IoU de 0,5 a 0,95, exigindo caixas cada vez mais justas | Mais rigorosa, sempre menor que o mAP50 |
| **Perdas** | Erro na posição da caixa, na presença de objeto e na classe | Quanto menor, melhor; a distância entre treino e validação indica overfitting |

A tabela usa a **melhor época** de cada simulação, a mesma escolhida pelo YOLOv5 para o `best.pt`. O tempo de treino inclui a preparação que o `train.py` faz antes da primeira época.

In [ ]:
def ler_resultados(pasta):
    '''Lê o results.csv do YOLOv5, removendo os espaços dos nomes das colunas.'''
    df = pd.read_csv(pasta / "results.csv")
    df.columns = [coluna.strip() for coluna in df.columns]
    return df

RESULTADOS = {epocas: ler_resultados(treino["pasta"]) for epocas, treino in TREINOS.items()}

linhas = []
for epocas, df in RESULTADOS.items():
    # Critério do YOLOv5 para escolher o best.pt: 10% do mAP50 + 90% do mAP50-95
    fitness = 0.1 * df["metrics/mAP_0.5"] + 0.9 * df["metrics/mAP_0.5:0.95"]
    melhor, final = df.loc[fitness.idxmax()], df.iloc[-1]
    linhas.append({
        "Simulação": f"{epocas} épocas",
        "Tempo de treino (min)": TREINOS[epocas]["segundos"] / 60,
        "Tempo por época (s)": TREINOS[epocas]["segundos"] / epocas,
        "Melhor época": int(melhor["epoch"]) + 1,
        "Precisão": melhor["metrics/precision"],
        "Recall": melhor["metrics/recall"],
        "mAP50": melhor["metrics/mAP_0.5"],
        "mAP50-95": melhor["metrics/mAP_0.5:0.95"],
        "Perda de caixa no treino (final)": final["train/box_loss"],
        "Perda de caixa na validação (final)": final["val/box_loss"],
    })
TABELA_EPOCAS = pd.DataFrame(linhas).set_index("Simulação")
TABELA_EPOCAS.to_csv(SAIDAS / "comparacao_epocas.csv")
display(TABELA_EPOCAS.round(3))

In [ ]:
# Curvas de treino e validação das duas simulações sobrepostas
metricas = [
    ("train/box_loss", "Perda de caixa | treino"), ("val/box_loss", "Perda de caixa | validação"),
    ("train/obj_loss", "Perda de objeto | treino"), ("val/obj_loss", "Perda de objeto | validação"),
    ("metrics/precision", "Precisão | validação"), ("metrics/recall", "Recall | validação"),
    ("metrics/mAP_0.5", "mAP50 | validação"), ("metrics/mAP_0.5:0.95", "mAP50-95 | validação"),
]
figura, eixos = plt.subplots(2, 4, figsize=(18, 7.5))
for eixo, (coluna, titulo) in zip(eixos.ravel(), metricas):
    for (epocas, df), cor in zip(RESULTADOS.items(), ["#E80A6E", "#2B6CB0"]):
        eixo.plot(df["epoch"] + 1, df[coluna], label=f"{epocas} épocas", color=cor, linewidth=2)
    eixo.set_title(titulo, fontsize=11)
    eixo.set_xlabel("Época")
eixos[0, 0].legend()
figura.suptitle("Curvas de treinamento | 30 x 60 épocas", fontsize=14, fontweight="bold")
plt.tight_layout()
figura.savefig(SAIDAS / "curvas_30x60.png", bbox_inches="tight")
plt.show()

**[A PREENCHER COM OS RESULTADOS DA EXECUÇÃO]**

### 3.7 Etapa de validação

Durante o treino a validação acontece automaticamente ao fim de cada época. Aqui ela é repetida de forma isolada com o `val.py`, usando o `best.pt` de cada simulação, para obter as métricas **por classe** nas 8 imagens de validação.

In [ ]:
# Validação oficial do YOLOv5 com os melhores pesos de cada simulação, com métricas por classe
for epocas, treino in TREINOS.items():
    print(f"\n{'=' * 25} Validação | {epocas} épocas {'=' * 25}")
    executar(
        "val.py",
        "--weights", treino["pasta"] / "weights" / "best.pt",
        "--data", ARQUIVO_YAML,
        "--task", "val",             # conjunto de validação
        "--img", 640,
        "--device", DISPOSITIVO,
        "--workers", TRABALHADORES,
        "--project", YOLOV5 / "runs" / "val",
        "--name", f"exp_{epocas}ep",
        "--exist-ok",
        "--verbose",                 # mostra as métricas separadas por classe
    )

### 3.8 Etapa de teste

O `detect.py` aplica cada modelo nas **8 imagens de teste**, que não participaram nem do treino nem da escolha do melhor ponto. As imagens processadas, com as caixas e a confiança de cada detecção, são salvas em `yolov5/runs/detect/exp_30ep` e `yolov5/runs/detect/exp_60ep`, o equivalente ao `runs/detect/expX` citado no enunciado, com nomes fixos para identificar cada simulação.

São essas imagens que apresentamos ao cliente da FarmTech.

In [ ]:
PASTA_DETECCOES = YOLOV5 / "runs" / "detect"

# Detecção nas imagens de teste com o best.pt de cada simulação
for epocas, treino in TREINOS.items():
    print(f"\n{'=' * 25} Teste | {epocas} épocas {'=' * 25}")
    executar(
        "detect.py",
        "--weights", treino["pasta"] / "weights" / "best.pt",
        "--source", DATASET / "images" / "test",   # as 8 imagens nunca vistas
        "--img", 640,
        "--conf-thres", LIMIAR_CONFIANCA,          # só desenha detecções com confiança a partir de 0,25
        "--device", DISPOSITIVO,
        "--project", PASTA_DETECCOES,
        "--name", f"exp_{epocas}ep",
        "--exist-ok",
        "--line-thickness", 3,
    )

#### Imagens de teste processadas pelos modelos

Cada linha mostra a mesma imagem de teste processada pelo modelo de 30 épocas (esquerda) e pelo de 60 épocas (direita).

In [ ]:
# Grade com as imagens de teste processadas: 30 épocas à esquerda e 60 épocas à direita de cada par
imagens_processadas, titulos = [], []
for caminho in sorted((DATASET / "images" / "test").glob("*.jpg")):
    for epocas in EPOCAS:
        imagens_processadas.append(Image.open(PASTA_DETECCOES / f"exp_{epocas}ep" / caminho.name))
        titulos.append(f"{caminho.stem} | {epocas} épocas")
grade_imagens(imagens_processadas, titulos, colunas=4, tamanho=4.2,
              titulo_geral="Teste da YOLOv5s customizada", arquivo="teste_yolov5_30x60.png")

#### Métricas no conjunto de teste

Para medir o teste com números, e usar a mesma régua na Entrega 2, todos os modelos passam pelo mesmo protocolo:

- **Detecção:** uma caixa prevista é um acerto quando tem a classe correta e IoU de pelo menos 0,5 com uma caixa real ainda não associada. Precisão e recall usam as detecções com confiança a partir de 0,25; o mAP50 usa todas as detecções.
- **Acurácia por imagem:** a classe da detecção mais confiante (a partir de 0,25) é a resposta do modelo para a imagem. Se nada for detectado, a resposta é "nenhuma" e conta como erro. É essa métrica que permite comparar detectores com a CNN, que só classifica.

In [ ]:
def rotulos_em_pixels(split):
    '''Converte os rótulos YOLO de um conjunto em caixas (classe, x1, y1, x2, y2) em pixels.'''
    verdade = {}
    for caminho in sorted((DATASET / "images" / split).glob("*.jpg")):
        W, H = Image.open(caminho).size
        verdade[caminho] = [(c, (xc - w / 2) * W, (yc - h / 2) * H, (xc + w / 2) * W, (yc + h / 2) * H)
                            for c, xc, yc, w, h in ler_rotulos(DATASET / "labels" / split / f"{caminho.stem}.txt")]
    return verdade

def iou(a, b):
    '''Intersection over Union entre duas caixas (x1, y1, x2, y2).'''
    largura = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    altura = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    intersecao = largura * altura
    uniao = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - intersecao
    return intersecao / uniao if uniao > 0 else 0.0

def average_precision(acertos, confiancas, total_reais):
    '''AP como área sob a curva precisão x recall, com o envelope de precisão (padrão VOC/COCO).'''
    if total_reais == 0 or not acertos:
        return 0.0
    ordem = np.argsort(-np.asarray(confiancas))
    tp = np.cumsum(np.asarray(acertos)[ordem])
    fp = np.cumsum(1 - np.asarray(acertos)[ordem])
    recall = np.concatenate([[0.0], tp / total_reais, [1.0]])
    precisao = np.concatenate([[1.0], tp / (tp + fp), [0.0]])
    # Envelope: em cada ponto, a maior precisão obtida dali para frente
    precisao = np.maximum.accumulate(precisao[::-1])[::-1]
    return float(np.sum((recall[1:] - recall[:-1]) * precisao[1:]))

def avaliar(predicoes, verdade):
    '''Métricas de detecção e acurácia por imagem.

    predicoes: {caminho: [(classe, confianca, x1, y1, x2, y2), ...]} com todas as detecções do modelo.
    '''
    por_classe = {}
    for id_classe, classe in enumerate(CLASSES):
        acertos, confiancas, total_reais, tp_limiar, fp_limiar = [], [], 0, 0, 0
        for caminho, reais in verdade.items():
            reais_classe = [r[1:] for r in reais if r[0] == id_classe]
            total_reais += len(reais_classe)
            usados = set()
            # Da predição mais confiante para a menos, associa cada uma à caixa real livre de maior IoU
            for p in sorted((p for p in predicoes[caminho] if p[0] == id_classe), key=lambda p: -p[1]):
                melhor_iou, melhor_k = max(((iou(p[2:], r), k) for k, r in enumerate(reais_classe) if k not in usados),
                                           default=(0.0, None))
                acerto = melhor_iou >= LIMIAR_IOU
                if acerto:
                    usados.add(melhor_k)
                acertos.append(int(acerto))
                confiancas.append(p[1])
                if p[1] >= LIMIAR_CONFIANCA:
                    tp_limiar += acerto
                    fp_limiar += not acerto
        por_classe[NOMES_EXIBICAO[classe]] = {
            "Precisão": tp_limiar / (tp_limiar + fp_limiar) if tp_limiar + fp_limiar else 0.0,
            "Recall": tp_limiar / total_reais if total_reais else 0.0,
            "AP50": average_precision(acertos, confiancas, total_reais),
            "Objetos reais": total_reais,
        }
    por_imagem = []
    for caminho in verdade:
        validas = [p for p in predicoes[caminho] if p[1] >= LIMIAR_CONFIANCA]
        mais_confiante = max(validas, key=lambda p: p[1]) if validas else None
        real = caminho.stem.split("_")[0]
        prevista = CLASSES[mais_confiante[0]] if mais_confiante else "nenhuma"
        por_imagem.append({"imagem": caminho.stem, "classe real": real, "classe prevista": prevista,
                           "confiança": mais_confiante[1] if mais_confiante else 0.0,
                           "detecções": len(validas), "acertou": prevista == real})
    por_classe, por_imagem = pd.DataFrame(por_classe).T, pd.DataFrame(por_imagem)
    precisao, recall = por_classe["Precisão"].mean(), por_classe["Recall"].mean()
    resumo = {
        "Acurácia por imagem": por_imagem["acertou"].mean(),
        "Precisão": precisao,
        "Recall": recall,
        "F1": 2 * precisao * recall / (precisao + recall) if precisao + recall else 0.0,
        "mAP50": por_classe["AP50"].mean(),
    }
    return resumo, por_classe, por_imagem

In [ ]:
def carregar_yolov5(pesos, dispositivo="cpu"):
    '''Carrega um best.pt pelo torch.hub, a partir do repositório local do YOLOv5.'''
    modelo = torch.hub.load(str(YOLOV5), "custom", path=str(pesos), source="local", _verbose=False, device=dispositivo)
    modelo.conf = 0.001  # mantém detecções de baixa confiança, necessárias para o cálculo do mAP
    modelo.iou = 0.45    # limiar da supressão de não máximos (NMS)
    return modelo

def prever_yolov5(modelo, caminho):
    '''Roda o modelo em uma imagem e devolve [(classe, confiança, x1, y1, x2, y2), ...].'''
    deteccoes = modelo(str(caminho), size=640).xyxy[0].cpu().numpy()
    return [(int(c), float(conf), x1, y1, x2, y2) for x1, y1, x2, y2, conf, c in deteccoes]

VERDADE_TESTE = rotulos_em_pixels("test")
IMAGENS_TESTE = list(VERDADE_TESTE)
DISPOSITIVO_HUB = "cuda:0" if torch.cuda.is_available() else "cpu"
AVALIACOES = {}

# Avaliação das duas simulações nas 8 imagens de teste
for epocas, treino in TREINOS.items():
    modelo = carregar_yolov5(treino["pasta"] / "weights" / "best.pt", DISPOSITIVO_HUB)
    predicoes = {caminho: prever_yolov5(modelo, caminho) for caminho in IMAGENS_TESTE}
    AVALIACOES[f"YOLOv5s customizada ({epocas} épocas)"] = avaliar(predicoes, VERDADE_TESTE)

display(pd.DataFrame({nome: avaliacao[0] for nome, avaliacao in AVALIACOES.items()}).T.round(3))
for nome, (_, por_classe, por_imagem) in AVALIACOES.items():
    print(f"\n{nome} | métricas por classe e resposta em cada imagem")
    display(por_classe.round(3))
    display(por_imagem.round(3))

**[A PREENCHER COM OS RESULTADOS DA EXECUÇÃO]**

# 4. Parte 2 | Entrega 2: YOLO tradicional e CNN treinada do zero

Com a YOLOv5 customizada pronta, a mesma base é usada para comparar duas abordagens concorrentes:

| Abordagem | Treino necessário | Rótulo necessário | Saída |
|---|---|---|---|
| **YOLOv5s customizada** (Entrega 1) | Ajuste fino a partir do COCO | Caixas por objeto | Caixas e classe |
| **YOLOv3 tradicional** | Nenhum: pesos originais treinados no COCO | Nenhum | Caixas e classe, entre as 80 classes COCO |
| **CNN treinada do zero** | Treino completo, sem pesos prévios | Só a classe da imagem | Classe da imagem inteira |

As três são avaliadas nas **mesmas 8 imagens de teste**, segundo os critérios do enunciado: precisão, tempo de treinamento, tempo de inferência e facilidade de uso e integração.

## 4.1 YOLO tradicional

A YOLO tradicional é a **YOLOv3 original do Darknet**, com os pesos oficiais treinados na base COCO, a mesma apresentada no capítulo sobre técnicas de detecção e segmentação. Ela não passa por nenhum treino com as nossas imagens: reconhece as 80 classes do COCO, entre elas `apple` e `scissors`.

Em vez de compilar o Darknet, a rede é carregada pelo módulo DNN do OpenCV, que lê os mesmos arquivos `yolov3.cfg` e `yolov3.weights`. O modelo é idêntico, e assim é possível medir o tempo de inferência imagem a imagem dentro do notebook. Na avaliação, `apple` corresponde a maçã e `scissors` a tesoura; as demais classes COCO ficam fora das métricas, mas aparecem na tabela do que a rede enxergou.

In [ ]:
import cv2
import urllib.request

# Arquivos da YOLOv3 original do Darknet, os mesmos do capítulo de detecção e segmentação
PASTA_MODELOS = RAIZ / "_modelos"
PASTA_MODELOS.mkdir(exist_ok=True)
ARQUIVOS_YOLOV3 = {
    "yolov3.cfg": "https://raw.githubusercontent.com/pjreddie/darknet/master/cfg/yolov3.cfg",
    "coco.names": "https://raw.githubusercontent.com/pjreddie/darknet/master/data/coco.names",
    "yolov3.weights": "https://pjreddie.com/media/files/yolov3.weights",
}
for arquivo, url in ARQUIVOS_YOLOV3.items():
    destino = PASTA_MODELOS / arquivo
    if not destino.exists():
        print(f"Baixando {arquivo}...")
        urllib.request.urlretrieve(url, destino)
    print(f"{arquivo}: {destino.stat().st_size / 1e6:.1f} MB")

COCO = (PASTA_MODELOS / "coco.names").read_text().strip().splitlines()
# Correspondência entre as classes COCO e as classes do projeto
COCO_PARA_PROJETO = {COCO.index("apple"): 0, COCO.index("scissors"): 1}
print(f"\nClasses COCO: {len(COCO)} | apple = índice {COCO.index('apple')} | scissors = índice {COCO.index('scissors')}")

In [ ]:
# Carrega a rede a partir dos bytes dos arquivos, o que também funciona em caminhos com acentos
rede_yolov3 = cv2.dnn.readNetFromDarknet(np.fromfile(str(PASTA_MODELOS / "yolov3.cfg"), np.uint8),
                                         np.fromfile(str(PASTA_MODELOS / "yolov3.weights"), np.uint8))
rede_yolov3.setPreferableBackend(cv2.dnn.DNN_BACKEND_OPENCV)
rede_yolov3.setPreferableTarget(cv2.dnn.DNN_TARGET_CPU)
CAMADAS_SAIDA = rede_yolov3.getUnconnectedOutLayersNames()

def ler_imagem_cv2(caminho):
    '''Lê uma imagem com OpenCV aceitando caminhos com acentos.'''
    return cv2.imdecode(np.fromfile(str(caminho), np.uint8), cv2.IMREAD_COLOR)

def prever_yolov3_coco(caminho, confianca_minima=0.001):
    '''Roda a YOLOv3 COCO e devolve [(id_coco, confiança, x1, y1, x2, y2), ...] após o NMS.'''
    imagem = ler_imagem_cv2(caminho)
    H, W = imagem.shape[:2]
    # Pré-processamento padrão da YOLOv3: pixels entre 0 e 1, 416 x 416 pixels, canais em RGB
    rede_yolov3.setInput(cv2.dnn.blobFromImage(imagem, 1 / 255.0, (416, 416), swapRB=True, crop=False))
    saidas = np.vstack(rede_yolov3.forward(CAMADAS_SAIDA))
    # Cada linha: centro x, centro y, largura, altura, objetividade e a pontuação das 80 classes
    linhas, ids = np.where(saidas[:, 5:] >= confianca_minima)
    confiancas = saidas[linhas, 5 + ids]
    xc, yc, w, h = (saidas[linhas, :4] * [W, H, W, H]).T
    caixas = np.stack([xc - w / 2, yc - h / 2, w, h], axis=1)
    deteccoes = []
    # Supressão de não máximos separada por classe
    for id_coco in np.unique(ids):
        indices = np.where(ids == id_coco)[0]
        mantidos = cv2.dnn.NMSBoxes(caixas[indices].tolist(), confiancas[indices].tolist(), confianca_minima, 0.45)
        for k in np.array(mantidos).flatten():
            x, y, bw, bh = caixas[indices[k]]
            deteccoes.append((int(id_coco), float(confiancas[indices[k]]), x, y, x + bw, y + bh))
    return deteccoes

# Detecções COCO completas e a versão filtrada para as duas classes do projeto
DETECCOES_COCO = {caminho: prever_yolov3_coco(caminho) for caminho in IMAGENS_TESTE}
predicoes_yolov3 = {caminho: [(COCO_PARA_PROJETO[d[0]], *d[1:]) for d in deteccoes if d[0] in COCO_PARA_PROJETO]
                    for caminho, deteccoes in DETECCOES_COCO.items()}
NOME_YOLOV3 = "YOLOv3 tradicional (COCO)"
AVALIACOES[NOME_YOLOV3] = avaliar(predicoes_yolov3, VERDADE_TESTE)

resumo_yolov3, por_classe_yolov3, por_imagem_yolov3 = AVALIACOES[NOME_YOLOV3]
display(pd.Series(resumo_yolov3, name=NOME_YOLOV3).to_frame().T.round(3))
display(por_classe_yolov3.round(3))

# O que a YOLOv3 enxergou em cada imagem, entre todas as 80 classes COCO
vistos = []
for caminho, deteccoes in DETECCOES_COCO.items():
    confiaveis = sorted((d for d in deteccoes if d[1] >= LIMIAR_CONFIANCA), key=lambda d: -d[1])
    vistos.append({"imagem": caminho.stem,
                   "detecções COCO (confiança)": ", ".join(f"{COCO[d[0]]} ({d[1]:.2f})" for d in confiaveis[:4]) or "nenhuma"})
display(por_imagem_yolov3.merge(pd.DataFrame(vistos), on="imagem").round(3))

In [ ]:
def desenhar_deteccoes(caminho, caixas, espessura=4):
    '''Desenha caixas [(texto, cor, x1, y1, x2, y2), ...] com legenda sobre a imagem.'''
    imagem = Image.open(caminho).convert("RGB")
    desenho = ImageDraw.Draw(imagem)
    fonte = ImageFont.load_default(size=18)
    for texto, cor, x1, y1, x2, y2 in caixas:
        desenho.rectangle([x1, y1, x2, y2], outline=cor, width=espessura)
        esquerda, topo, direita, base = desenho.textbbox((x1, y1), texto, font=fonte)
        desenho.rectangle([x1, y1 - (base - topo) - 6, x1 + (direita - esquerda) + 8, y1], fill=cor)
        desenho.text((x1 + 4, y1 - (base - topo) - 5), texto, font=fonte, fill="white")
    return imagem

# Imagens de teste com tudo o que a YOLOv3 detectou: classes do projeto em destaque e demais classes COCO em cinza
imagens_yolov3, titulos_yolov3 = [], []
for caminho, deteccoes in DETECCOES_COCO.items():
    caixas = []
    for id_coco, conf, x1, y1, x2, y2 in deteccoes:
        if conf < LIMIAR_CONFIANCA:
            continue
        cor = CORES[CLASSES[COCO_PARA_PROJETO[id_coco]]] if id_coco in COCO_PARA_PROJETO else (110, 110, 120)
        caixas.append((f"{COCO[id_coco]} {conf:.2f}", cor, x1, y1, x2, y2))
    imagens_yolov3.append(desenhar_deteccoes(caminho, caixas))
    titulos_yolov3.append(caminho.stem)
grade_imagens(imagens_yolov3, titulos_yolov3, colunas=4, tamanho=4,
              titulo_geral="Teste da YOLOv3 tradicional (COCO, sem treino)", arquivo="teste_yolov3.png")

**[A PREENCHER COM OS RESULTADOS DA EXECUÇÃO]**

## 4.2 CNN treinada do zero

Uma CNN classifica a **imagem inteira**, então o rótulo precisou ser refeito: em vez das caixas do Make Sense, cada imagem recebe apenas a sua classe, obtida do nome do arquivo (`maca_xx` = 0, `tesoura_xx` = 1).

Arquitetura inspirada no modelo montado à mão no capítulo de CNN, adaptada a um dataset pequeno:

| Componente | Escolha | Motivo |
|---|---|---|
| Entrada | 150 x 150 pixels, RGB | Mesmo tamanho usado no capítulo; leve o bastante para treinar rápido |
| Aumento de dados | Espelhamento, rotação, zoom, translação e contraste aleatórios | Com 64 imagens de treino, cria variações a cada época e reduz a memorização |
| Extração de características | 4 blocos `Conv2D` + `MaxPooling2D`, com 16, 32, 64 e 64 filtros | Os filtros aprendem de bordas simples até formas do objeto |
| Decisão | `GlobalAveragePooling2D`, `Dropout(0,3)`, `Dense(64)` e `Dense(2, softmax)` | Poucos parâmetros e regularização, adequados a pouco dado |
| Treino | Adam, até 150 épocas, parada antecipada com paciência de 25 épocas pela perda de validação | Interrompe quando a validação para de melhorar e restaura os melhores pesos |

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix, precision_recall_fscore_support

# Evita que o TensorFlow reserve toda a memória da GPU, compartilhada com o PyTorch
for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)
keras.utils.set_random_seed(SEMENTE)

TAMANHO_CNN = (150, 150)

def carregar_classificacao(split):
    '''Carrega um conjunto como matriz de pixels e a classe da imagem inteira (0 = maçã, 1 = tesoura).'''
    caminhos = sorted((DATASET / "images" / split).glob("*.jpg"))
    X = np.stack([np.asarray(Image.open(c).convert("RGB").resize(TAMANHO_CNN)) for c in caminhos]).astype("float32")
    y = np.array([CLASSES.index(c.stem.split("_")[0]) for c in caminhos])
    return X, y, caminhos

X_treino, y_treino, _ = carregar_classificacao("train")
X_val, y_val, _ = carregar_classificacao("val")
X_teste, y_teste, CAMINHOS_TESTE_CNN = carregar_classificacao("test")

for nome, X, y in [("Treino", X_treino, y_treino), ("Validação", X_val, y_val), ("Teste", X_teste, y_teste)]:
    print(f"{nome:<10} {X.shape} | maçãs: {(y == 0).sum()} | tesouras: {(y == 1).sum()}")
print(f"TensorFlow {tf.__version__} | GPU: {bool(tf.config.list_physical_devices('GPU'))}")

In [ ]:
# Camadas de aumento de dados: só atuam durante o treino
aumento = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.2),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomContrast(0.2),
], name="aumento_de_dados")

cnn = keras.Sequential([
    keras.Input(shape=(*TAMANHO_CNN, 3)),
    aumento,
    layers.Rescaling(1.0 / 255),                               # pixels de 0-255 para 0-1
    layers.Conv2D(16, 3, padding="same", activation="relu"),   # bloco 1: bordas e cores
    layers.MaxPooling2D(),
    layers.Conv2D(32, 3, padding="same", activation="relu"),   # bloco 2: texturas
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, padding="same", activation="relu"),   # bloco 3: partes do objeto
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, padding="same", activation="relu"),   # bloco 4: formas mais completas
    layers.MaxPooling2D(),
    layers.GlobalAveragePooling2D(),                           # resume cada mapa de características em um valor
    layers.Dropout(0.3),                                       # desliga 30% das ativações no treino
    layers.Dense(64, activation="relu"),                       # camada oculta de decisão
    layers.Dense(len(CLASSES), activation="softmax"),          # probabilidade de cada classe
], name="cnn_farmtech")

cnn.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
            loss="sparse_categorical_crossentropy", metrics=["accuracy"])
cnn.summary()

In [ ]:
EPOCAS_CNN = 150

# Para o treino quando a perda de validação não melhora por 25 épocas e restaura os melhores pesos
parada_antecipada = keras.callbacks.EarlyStopping(monitor="val_loss", patience=25, restore_best_weights=True)

inicio = time.perf_counter()
historico = cnn.fit(X_treino, y_treino, validation_data=(X_val, y_val), epochs=EPOCAS_CNN, batch_size=16,
                    callbacks=[parada_antecipada], verbose=2)
TEMPO_TREINO_CNN = time.perf_counter() - inicio

EPOCAS_EXECUTADAS_CNN = len(historico.history["loss"])
MELHOR_EPOCA_CNN = int(np.argmin(historico.history["val_loss"])) + 1
print(f"\nTreino da CNN: {TEMPO_TREINO_CNN:.1f} s | {EPOCAS_EXECUTADAS_CNN} épocas executadas | melhor época: {MELHOR_EPOCA_CNN}")

In [ ]:
# Curvas de perda e acurácia da CNN
hist = pd.DataFrame(historico.history)
figura, eixos = plt.subplots(1, 2, figsize=(13, 4))
for eixo, (metrica, titulo) in zip(eixos, [("loss", "Perda"), ("accuracy", "Acurácia")]):
    eixo.plot(hist.index + 1, hist[metrica], label="Treino", color="#E80A6E", linewidth=2)
    eixo.plot(hist.index + 1, hist[f"val_{metrica}"], label="Validação", color="#2B6CB0", linewidth=2)
    eixo.axvline(MELHOR_EPOCA_CNN, color="gray", linestyle="--", label="Melhor época")
    eixo.set_title(f"{titulo} | CNN do zero")
    eixo.set_xlabel("Época")
    eixo.legend()
plt.tight_layout()
figura.savefig(SAIDAS / "curvas_cnn.png", bbox_inches="tight")
plt.show()

In [ ]:
# Avaliação da CNN nas 8 imagens de teste
perda_teste, acuracia_teste = cnn.evaluate(X_teste, y_teste, verbose=0)
probabilidades = cnn.predict(X_teste, verbose=0)
previstas = probabilidades.argmax(axis=1)
nomes = [NOMES_EXIBICAO[c] for c in CLASSES]

print(f"Perda no teste: {perda_teste:.3f} | Acurácia no teste: {acuracia_teste:.3f}\n")
print(classification_report(y_teste, previstas, labels=[0, 1], target_names=nomes, zero_division=0))

figura, eixo = plt.subplots(figsize=(4.5, 4))
ConfusionMatrixDisplay(confusion_matrix(y_teste, previstas, labels=[0, 1]), display_labels=nomes).plot(
    ax=eixo, cmap="RdPu", colorbar=False)
eixo.set_title("Matriz de confusão | CNN no teste")
eixo.grid(False)
plt.tight_layout()
figura.savefig(SAIDAS / "matriz_confusao_cnn.png", bbox_inches="tight")
plt.show()

# Registro no mesmo formato das outras abordagens (precisão e recall de classificação, média entre classes)
precisao_cnn, recall_cnn, f1_cnn, _ = precision_recall_fscore_support(y_teste, previstas, labels=[0, 1],
                                                                       average="macro", zero_division=0)
por_imagem_cnn = pd.DataFrame({
    "imagem": [c.stem for c in CAMINHOS_TESTE_CNN],
    "classe real": [CLASSES[i] for i in y_teste],
    "classe prevista": [CLASSES[i] for i in previstas],
    "confiança": probabilidades.max(axis=1),
    "acertou": previstas == y_teste,
})
NOME_CNN = "CNN treinada do zero"
AVALIACOES[NOME_CNN] = ({"Acurácia por imagem": float(acuracia_teste), "Precisão": precisao_cnn, "Recall": recall_cnn,
                         "F1": f1_cnn, "mAP50": np.nan}, None, por_imagem_cnn)
display(por_imagem_cnn.round(3))

# Imagens de teste com a classe prevista pela CNN
grade_imagens(
    [Image.open(c) for c in CAMINHOS_TESTE_CNN],
    [f"{c.stem} | {NOMES_EXIBICAO[CLASSES[p]]} ({prob.max():.2f}) {'ok' if p == y else 'ERRO'}"
     for c, p, prob, y in zip(CAMINHOS_TESTE_CNN, previstas, probabilidades, y_teste)],
    colunas=4, tamanho=3.6, titulo_geral="Teste da CNN treinada do zero", arquivo="teste_cnn.png",
)

**[A PREENCHER COM OS RESULTADOS DA EXECUÇÃO]**

## 4.3 Tempo de inferência

O tempo é medido **de ponta a ponta por imagem**: leitura do arquivo, pré-processamento, inferência e pós-processamento, uma imagem por vez, que é como o sistema funcionaria recebendo fotos de uma câmera. Cada medição faz 3 passagens de aquecimento e depois percorre as 8 imagens de teste 5 vezes, com o limiar de confiança de 0,25.

Todas as abordagens são medidas em **CPU**, para uma comparação no mesmo hardware. Quando há GPU, YOLOv5 e CNN também são medidas nela; a YOLOv3 roda pelo OpenCV instalado via pip, que não tem suporte a CUDA.

In [ ]:
def tempo_medio_ms(funcao, caminhos, aquecimento=3, repeticoes=5):
    '''Tempo médio por imagem, em milissegundos, após algumas execuções de aquecimento.'''
    for caminho in caminhos[:aquecimento]:
        funcao(caminho)
    inicio = time.perf_counter()
    for _ in range(repeticoes):
        for caminho in caminhos:
            funcao(caminho)
    return (time.perf_counter() - inicio) * 1000 / (repeticoes * len(caminhos))

TEMPOS = {}
dispositivos_torch = [("CPU", "cpu")] + ([("GPU", "cuda:0")] if torch.cuda.is_available() else [])

# YOLOv5 customizada, nas duas simulações
for epocas, treino in TREINOS.items():
    for rotulo, dispositivo in dispositivos_torch:
        modelo = carregar_yolov5(treino["pasta"] / "weights" / "best.pt", dispositivo)
        modelo.conf = LIMIAR_CONFIANCA
        TEMPOS[(f"YOLOv5s customizada ({epocas} épocas)", rotulo)] = tempo_medio_ms(
            lambda caminho: prever_yolov5(modelo, caminho), IMAGENS_TESTE)

# YOLOv3 tradicional pelo OpenCV DNN
TEMPOS[(NOME_YOLOV3, "CPU")] = tempo_medio_ms(lambda caminho: prever_yolov3_coco(caminho, LIMIAR_CONFIANCA), IMAGENS_TESTE)

def prever_cnn(caminho):
    '''Lê, redimensiona e classifica uma imagem com a CNN.'''
    x = np.asarray(Image.open(caminho).convert("RGB").resize(TAMANHO_CNN), dtype="float32")[None]
    return cnn(x, training=False)

# CNN treinada do zero
dispositivos_tf = [("CPU", "/CPU:0")] + ([("GPU", "/GPU:0")] if tf.config.list_physical_devices("GPU") else [])
for rotulo, dispositivo in dispositivos_tf:
    with tf.device(dispositivo):
        TEMPOS[(NOME_CNN, rotulo)] = tempo_medio_ms(prever_cnn, IMAGENS_TESTE)

TABELA_TEMPOS = pd.Series(TEMPOS).unstack()
TABELA_TEMPOS.columns = [f"Inferência {coluna} (ms/imagem)" for coluna in TABELA_TEMPOS.columns]
display(TABELA_TEMPOS.round(1))

## 4.4 Comparação consolidada

Para a CNN, precisão e recall são métricas de classificação (média entre as classes) e o mAP50 não se aplica. O tempo de treino da YOLOv3 é zero porque ela usa pesos prontos: o custo do treino original no COCO não entra na conta.

In [ ]:
# Tabela única com precisão, tempo de treino e tempo de inferência das quatro configurações
linhas = []
for nome, (resumo, _, _) in AVALIACOES.items():
    if nome.startswith("YOLOv5"):
        segundos_treino = TREINOS[int(nome.split("(")[1].split()[0])]["segundos"]
    elif nome == NOME_CNN:
        segundos_treino = TEMPO_TREINO_CNN
    else:
        segundos_treino = 0.0
    linhas.append({"Abordagem": nome, **resumo, "Tempo de treino (s)": segundos_treino})
COMPARACAO = pd.DataFrame(linhas).set_index("Abordagem").join(TABELA_TEMPOS)
COMPARACAO.to_csv(SAIDAS / "comparacao_abordagens.csv")
display(COMPARACAO.round(3))

In [ ]:
# Gráficos de barras dos três critérios numéricos
cores = ["#F58BB8", "#E80A6E", "#8C8C96", "#2B6CB0"]
rotulos = ["YOLOv5s\n30 épocas", "YOLOv5s\n60 épocas", "YOLOv3\ntradicional", "CNN\ndo zero"]
coluna_cpu = "Inferência CPU (ms/imagem)"
figura, eixos = plt.subplots(1, 3, figsize=(18, 5))
graficos = [
    ("Acurácia por imagem", "Acurácia por imagem no teste", "{:.0%}"),
    ("Tempo de treino (s)", "Tempo de treino (s)", "{:.0f} s"),
    (coluna_cpu, "Inferência em CPU (ms por imagem)", "{:.0f} ms"),
]
for eixo, (coluna, titulo, formato) in zip(eixos, graficos):
    valores = COMPARACAO[coluna].values
    barras = eixo.bar(rotulos, valores, color=cores)
    for barra, valor in zip(barras, valores):
        eixo.annotate(formato.format(valor), (barra.get_x() + barra.get_width() / 2, barra.get_height()),
                      ha="center", va="bottom", fontsize=11, fontweight="bold")
    eixo.set_title(titulo, fontsize=12, fontweight="bold")
    eixo.margins(y=0.15)
eixos[0].set_ylim(0, 1.15)
plt.tight_layout()
figura.savefig(SAIDAS / "comparacao_abordagens.png", bbox_inches="tight")
plt.show()

## 4.5 Facilidade de uso e integração

| Aspecto | YOLOv5s customizada | YOLOv3 tradicional | CNN treinada do zero |
|---|---|---|---|
| Preparação dos dados | Rotular caixas em todas as imagens | Nenhuma | Só a classe de cada imagem |
| Treinamento | Um comando (`train.py`), poucos minutos na GPU | Nenhum | Definir arquitetura, compilar e treinar |
| Uso no código | `detect.py` ou `torch.hub.load`, em poucas linhas | Carregar `cfg` e `weights` e decodificar a saída da rede à mão | `model.predict`, em poucas linhas |
| Classes reconhecidas | Exatamente as do projeto | 80 classes fixas do COCO | Exatamente as do projeto |
| Localização do objeto | Sim | Sim | Não |
| Dependências | PyTorch e repositório do YOLOv5 | OpenCV e arquivos do Darknet | TensorFlow e Keras |
| Integração com câmera ou ESP32-CAM | Direta: aceita imagem, vídeo e *stream* | Possível, com código próprio de captura e desenho | Possível, mas não indica onde está o objeto |

**[A PREENCHER COM OS RESULTADOS DA EXECUÇÃO]**

**[A PREENCHER COM OS RESULTADOS DA EXECUÇÃO]**

In [ ]:
if EM_COLAB:
    # Copia pesos, curvas e imagens processadas do YOLOv5 para o Google Drive
    shutil.copytree(YOLOV5 / "runs", PASTA_DRIVE / "runs", dirs_exist_ok=True)
    print(f"Resultados do YOLOv5 copiados para {PASTA_DRIVE / 'runs'}")
print(f"Tabelas e figuras em {SAIDAS}")

## 6. Referências

- FIAP. *ESP32: a janela para a visão computacional*. Fase 6, Capítulo 3, 2025.
- FIAP. *Desbravando o Deep Learning: primeiros passos com CNN*. Fase 6, Capítulo 9, 2024.
- FIAP. *O olhar digital: técnicas de detecção e segmentação*. Fase 6, Capítulo 10, 2025.
- JOCHER, G. et al. *Ultralytics YOLOv5*. GitHub, 2020. Disponível em: https://github.com/ultralytics/yolov5
- REDMON, J.; FARHADI, A. *YOLOv3: an incremental improvement*. arXiv:1804.02767, 2018.
- LIN, T.-Y. et al. *Microsoft COCO: common objects in context*. ECCV, 2014.
- KUZNETSOVA, A. et al. *The Open Images Dataset V4*. International Journal of Computer Vision, 2020.
- MAKE SENSE. Ferramenta de rotulação de imagens. Disponível em: https://www.makesense.ai